# FoxTrack 01: Kaggle setup and ETL

This notebook converts the Kitsune CSV files to Parquet with Spark and finds where each attack starts.

**Before you run it (right panel of the Kaggle editor):**

1. **Input > Add Input:** add the dataset `ymirsky/network-attack-dataset-kitsune`.
2. **Settings > Internet:** set to on. The notebook installs PySpark and clones the code.
3. **Private repository only:** in **Add-ons > Secrets**, add a secret with the name `GITHUB_TOKEN` and attach it to this notebook.
4. An accelerator (GPU) is not necessary.

In [ ]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
INPUT_ROOT = "/kaggle/input"
OUT_DIR = "/kaggle/working/kitsune_parquet"

In [ ]:
# Spark needs Java. Install it only if it is not there.
import shutil, subprocess
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
subprocess.run(["java", "-version"], check=True)
!pip install -q pyspark mlflow

In [ ]:
# Get the project code.
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
if not os.path.exists("/kaggle/working/repo"):
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
         f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
        capture_output=True, text=True,
    )
    if done.returncode:
        raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import etl

## 1. Find the files

Examine this table before the conversion. There must be nine captures, and each must have one feature file and one label file.

In [ ]:
import pandas as pd
captures = etl.discover(INPUT_ROOT)
files = pd.DataFrame([{
    "attack": c.name,
    "features_file": c.features_path.name,
    "features_gb": round(c.features_path.stat().st_size / 1e9, 2),
    "labels_file": c.labels_path.name,
    **etl.sniff_features(c.features_path),
} for c in captures])
print("Total size of feature files (GB):", files["features_gb"].sum().round(2))
files

## 2. Convert to Parquet

Each capture is one Spark task. Four captures run at the same time. The values are stored as 32-bit floats to stay below the 20 GB limit of `/kaggle/working`.

The conversion stops with an error if the number of feature rows and the number of labels are different.

In [ ]:
%%time
spark = etl.build_spark(driver_memory="20g")
summary = etl.convert_all(spark, INPUT_ROOT, OUT_DIR, workers=4)
summary

## 3. Examine the result

Important columns in the summary:

- `features_as_expected`: must be `True` (115 features).
- `first_attack_idx`: the packet where the attack starts. All packets before it are benign.
- `episodes`: the number of attack periods. This number controls the train/test split design.
- `null_f0_rows`: must be 0. A different value shows rows that Spark could not read.

In [ ]:
!du -sh {OUT_DIR}/features
df = spark.read.parquet(f"{OUT_DIR}/features")
print("Rows:", df.count(), "| Columns:", len(df.columns))
df.groupBy("attack").pivot("label").count().orderBy("attack").show()

## 4. Keep the output

Select **Save Version > Save & Run All (Commit)**. Kaggle then keeps `kitsune_parquet` as the notebook output.
In subsequent notebooks, use **Add Input > Notebook Output** to attach it. Then you do not need to run the conversion again.